## **Importing Necessary Libraries**

In [1]:
import numpy as np
import keras
import os
from pathlib import Path

2024-08-15 19:33:36.211671: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-15 19:33:36.211765: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-15 19:33:36.350447: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## **Setting Hyperparameters and Data Paths**

In [2]:
batch_size = 64  # Batch size for training.
epochs = 100  # Number of epochs to train for.
latent_dim = 256  # Latent dimensionality of the encoding space.
num_samples = 10000  # Number of samples to train on.

# Path to the data txt file on disk.
data_path = "/kaggle/input/english-france-dictionary/fra.txt"

## **Data Preparation and Vectorization**

In [3]:
# Containers for input and target sequences and characters
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

# Read the data file and split it into lines
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
    
# Process each line (each line contains a source-target pair)
for line in lines[: min(num_samples, len(lines) - 1)]:
    # Split the line into input (English) and target (French) sentences
    input_text, target_text, _ = line.split("\t")
    
    # Add start and end tokens to the target sentence
    target_text = "\t" + target_text + "\n"
    input_texts.append(input_text)
    target_texts.append(target_text)
    
    # Update the sets of unique characters in the input and target texts
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

## **Defining Character Sets and Sequence Lengths**

In [4]:
# Sort the characters to have a consistent index
input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

Number of samples: 10000
Number of unique input tokens: 70
Number of unique output tokens: 91
Max sequence length for inputs: 14
Max sequence length for outputs: 59


## **Creating Character Indexes**

In [5]:
# Create dictionaries mapping characters to their index in the sorted list
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

## **Initializing Encoder and Decoder Input Data**

In [6]:
# Initialize 3D numpy arrays to hold one-hot encoded data for encoder inputs, decoder inputs, and decoder targets
encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32",
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)

In [7]:
# Populate the one-hot encoded arrays
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    # One-hot encode the input sequence
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0
    
    # One-hot encode the target sequence for both decoder input and target output
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data is one step ahead of decoder_input_data
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0
    
    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0

## **Building the Sequence-to-Sequence Model**

In [8]:
# Define the encoder model
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
# Only the encoder states are kept; the outputs are discarded
encoder_states = [state_h, state_c]

# Define the decoder model
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))
# The decoder uses the encoder states as its initial state
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
# The output layer that generates predictions for each timestep
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Combine encoder and decoder into a training model
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the model with RMSprop optimizer and categorical crossentropy loss
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

## **Training the Model**

In [9]:
# Train the model
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
)

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.7069 - loss: 1.5424 - val_accuracy: 0.7141 - val_loss: 1.1691
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7478 - loss: 0.9508 - val_accuracy: 0.7248 - val_loss: 0.9606
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7630 - loss: 0.8593 - val_accuracy: 0.7426 - val_loss: 0.9252
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7868 - loss: 0.7749 - val_accuracy: 0.7769 - val_loss: 0.7761
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8043 - loss: 0.6824 - val_accuracy: 0.7892 - val_loss: 0.7214
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8147 - loss: 0.6394 - val_accuracy: 0.7989 - val_loss: 0.6867
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8231 - loss: 0.6041 - val_accuracy: 0.8045 - val_loss: 0.6854
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.8297 - loss: 0.5842 - 

## **Building the Inference Models**

In [10]:
# Define the encoder model for inference
encoder_model = keras.Model(encoder_inputs, encoder_states)

# Define the decoder model for inference
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)


In [11]:
# Reverse-lookup token index to decode sequences back to something readable
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

## **Defining the Decoding Function**

In [12]:
# Function to decode a sequence
def decode_sequence(input_seq):
    # Encode the input as state vectors
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for generating output sequence
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        # Predict the next character
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value, verbose=0)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length or find stop character
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1)
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]

    return decoded_sentence

## **Testing the Model**

In [13]:
# Decode and print the first 20 sequences from the training set
for seq_index in range(20):
    # Take one sequence (part of the training set) for trying out decoding
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: Go.
Decoded sentence: Continuez !

-
Input sentence: Go.
Decoded sentence: Continuez !

-
Input sentence: Go.
Decoded sentence: Continuez !

-
Input sentence: Go.
Decoded sentence: Continuez !

-
Input sentence: Hi.
Decoded sentence: Salut !

-
Input sentence: Hi.
Decoded sentence: Salut !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run!
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

-
Input sentence: Run.
Decoded sentence: File !

